# Observability & Troubleshooting - Comprehensive Notes

## Table of Contents
1. [Introduction to Observability](#introduction-to-observability)
2. [The Three Pillars of Observability](#the-three-pillars-of-observability)
3. [Logs](#logs)
4. [Metrics](#metrics)
5. [Traces](#traces)
6. [OpenTelemetry](#opentelemetry)
7. [Debugging Production Failures](#debugging-production-failures)
8. [SLOs, SLIs, and SLAs](#slos-slis-and-slas)
9. [Observability Best Practices](#observability-best-practices)
10. [Tools and Ecosystem](#tools-and-ecosystem)

---

## Introduction to Observability

### What is Observability?

**Observability** is the ability to understand the internal state of a system by examining its outputs (logs, metrics, traces).

**Key Principle**: *"You can ask questions about your system that you didn't anticipate when building it."*

### Observability vs Monitoring

| Aspect | Monitoring | Observability |
|--------|-----------|---------------|
| **Focus** | Known failures | Unknown failures |
| **Approach** | Pre-defined dashboards | Ad-hoc queries |
| **Questions** | "Is it broken?" | "Why is it broken?" |
| **Scope** | System health | System behavior |
| **Mindset** | Reactive | Proactive + Reactive |

**Example**:
- **Monitoring**: "CPU > 80% → Alert" (predefined threshold)
- **Observability**: "Why did request latency spike for users in Tokyo at 3 PM?" (exploratory investigation)

### Why Observability Matters

```text
Production Incident
       ↓
Without Observability:
- "Something is broken"
- Blind guessing
- Long MTTR (Mean Time To Recovery)
- User impact extends

With Observability:
- "API latency increased because DB connection pool exhausted"
- Root cause identified quickly
- Fast mitigation
- Minimal user impact
```

**Key Benefits**:
- ✅ Faster debugging
- ✅ Better system understanding
- ✅ Proactive issue detection
- ✅ Data-driven decisions
- ✅ Improved reliability

---

## The Three Pillars of Observability

### Overview

The three pillars work together to provide complete system visibility:

```text
┌─────────────────────────────────────────────────┐
│                  Observability                   │
├─────────────────┬─────────────────┬──────────────┤
│      LOGS       │     METRICS     │    TRACES    │
├─────────────────┼─────────────────┼──────────────┤
│  What happened  │ How much/often  │ Where/Why    │
│                 │                 │              │
│  Event records  │  Aggregated #s  │ Request flow │
│                 │                 │              │
│  Unstructured/  │  Time series    │ Distributed  │
│  Structured     │  data           │ context      │
└─────────────────┴─────────────────┴──────────────┘
```

### How They Work Together

**Example Scenario**: API request is slow

1. **Metrics** tell you: *"P95 latency increased from 100ms to 2000ms"*
2. **Traces** show you: *"Database query took 1.8s"*
3. **Logs** reveal: *"Connection pool exhausted at 14:30:45"*

**Together**: You know **what** happened, **when**, **where**, and **why**.

---

## Logs

### What are Logs?

**Logs** are timestamped records of discrete events that happened in your system.

**Types of Logs**:
- Application logs (your code)
- System logs (OS, kernel)
- Access logs (web servers)
- Audit logs (security, compliance)
- Error logs (exceptions, failures)

### Log Levels

Standard severity levels (from lowest to highest):

| Level | Use Case | Example |
|-------|----------|---------|
| **DEBUG** | Detailed info for debugging | `"User object created: {user_id: 123}"` |
| **INFO** | General informational messages | `"User logged in successfully"` |
| **WARNING** | Something unexpected but handled | `"Retry attempt 2/3 for API call"` |
| **ERROR** | Error occurred, function failed | `"Failed to process payment: timeout"` |
| **CRITICAL** | System-level failure | `"Database connection lost"` |

### Structured vs Unstructured Logs

#### Unstructured Logs (Traditional)
```text
2026-01-22 10:30:45 ERROR Failed to process order 12345 for user alice@example.com
```

**Problems**:
- ❌ Hard to parse
- ❌ Hard to search
- ❌ Hard to aggregate
- ❌ No context isolation

#### Structured Logs (Modern)
```json
{
  "timestamp": "2026-01-22T10:30:45.123Z",
  "level": "ERROR",
  "message": "Failed to process order",
  "order_id": 12345,
  "user_email": "alice@example.com",
  "error_type": "PaymentTimeout",
  "duration_ms": 5000,
  "service": "order-service",
  "trace_id": "abc123"
}
```

**Benefits**:
- ✅ Easy to query: `WHERE order_id = 12345`
- ✅ Easy to filter: `WHERE error_type = "PaymentTimeout"`
- ✅ Easy to aggregate: `COUNT(*) GROUP BY error_type`
- ✅ Machine-readable
- ✅ Preserves context

### Implementing Structured Logging in Python

#### Using Python's Built-in Logging

```python
import logging
import json
from datetime import datetime

class StructuredFormatter(logging.Formatter):
    def format(self, record):
        log_data = {
            "timestamp": datetime.utcnow().isoformat(),
            "level": record.levelname,
            "message": record.getMessage(),
            "logger": record.name,
            "module": record.module,
            "function": record.funcName,
        }
        
        # Add extra fields if provided
        if hasattr(record, "extra_fields"):
            log_data.update(record.extra_fields)
        
        return json.dumps(log_data)

# Configure logger
logger = logging.getLogger(__name__)
handler = logging.StreamHandler()
handler.setFormatter(StructuredFormatter())
logger.addHandler(handler)
logger.setLevel(logging.INFO)

# Usage
def process_order(order_id, user_id):
    logger.info(
        "Processing order",
        extra={
            "extra_fields": {
                "order_id": order_id,
                "user_id": user_id,
                "service": "order-service"
            }
        }
    )
```

#### Using structlog (Recommended)

```python
import structlog

# Configure structlog
structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.stdlib.add_logger_name,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer()
    ],
    context_class=dict,
    logger_factory=structlog.stdlib.LoggerFactory(),
)

logger = structlog.get_logger()

# Usage - much cleaner!
def process_order(order_id, user_id):
    logger.info(
        "order_processed",
        order_id=order_id,
        user_id=user_id,
        amount=99.99,
        currency="USD"
    )
    
# Output:
# {
#   "event": "order_processed",
#   "level": "info",
#   "timestamp": "2026-01-22T10:30:45.123Z",
#   "order_id": 12345,
#   "user_id": 67890,
#   "amount": 99.99,
#   "currency": "USD"
# }
```

### Log Context and Correlation

**The Challenge**: In distributed systems, a single user request might touch 10+ services. How do you connect the logs?

**Solution**: Correlation IDs (Trace IDs, Request IDs)

```python
import structlog
import uuid

def add_request_id(logger, method_name, event_dict):
    # Add request ID to all log entries
    if "request_id" not in event_dict:
        event_dict["request_id"] = str(uuid.uuid4())
    return event_dict

structlog.configure(
    processors=[
        add_request_id,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer()
    ]
)

logger = structlog.get_logger()

def handle_request(request):
    request_id = request.headers.get("X-Request-ID", str(uuid.uuid4()))
    
    # Bind request_id to logger context
    log = logger.bind(request_id=request_id)
    
    log.info("request_received", path=request.path)
    
    # Pass request_id to downstream services
    process_payment(request_id)
    update_inventory(request_id)
    
    log.info("request_completed", status=200)

def process_payment(request_id):
    log = logger.bind(request_id=request_id)
    log.info("payment_processing")
    # All logs will have the same request_id
```

**Now you can query**:
```sql
SELECT * FROM logs WHERE request_id = "abc-123-def"
```

And see the **entire request flow** across all services.

### Log Sampling and Volume Management

**Problem**: High-traffic systems generate too many logs (millions per second).

**Solutions**:

#### 1. Adaptive Sampling
```python
import random

def should_log(level, sample_rate=0.01):
    if level in ["ERROR", "CRITICAL"]:
        return True  # Always log errors
    return random.random() < sample_rate

def log_info(message, **kwargs):
    if should_log("INFO", sample_rate=0.01):  # Log 1% of INFO
        logger.info(message, **kwargs)
```

#### 2. Dynamic Log Levels
```python
# Normal operation: INFO
# Under high load: WARNING only
# During incident: DEBUG

import os

LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO")
logger.setLevel(LOG_LEVEL)
```

#### 3. Log Aggregation
- Aggregate similar logs
- Example: "Payment timeout" × 1000 → "Payment timeout occurred 1000 times in last minute"

### Best Practices for Logging

**Do**:
- ✅ Use structured logging
- ✅ Include correlation IDs
- ✅ Log at appropriate levels
- ✅ Include context (user_id, order_id, etc.)
- ✅ Log entry/exit of critical functions
- ✅ Log external API calls
- ✅ Log state changes
- ✅ Use consistent field names

**Don't**:
- ❌ Log sensitive data (passwords, credit cards, SSNs)
- ❌ Log in tight loops (causes performance issues)
- ❌ Log everything (signal-to-noise ratio)
- ❌ Use inconsistent formats
- ❌ Log without context
- ❌ Forget to handle PII (Personal Identifiable Information)

### Sensitive Data Handling

```python
import structlog
import re

def mask_sensitive_data(logger, method_name, event_dict):
    """Mask sensitive fields in logs"""
    sensitive_fields = ["password", "credit_card", "ssn", "api_key"]
    
    for key, value in event_dict.items():
        if key in sensitive_fields:
            if isinstance(value, str):
                event_dict[key] = "***REDACTED***"
        # Mask credit card numbers in any field
        if isinstance(value, str):
            event_dict[key] = re.sub(
                r'\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b',
                '****-****-****-****',
                value
            )
    
    return event_dict

structlog.configure(
    processors=[
        mask_sensitive_data,
        structlog.processors.JSONRenderer()
    ]
)

# Usage
logger.info(
    "user_login",
    username="alice",
    password="secret123",  # Will be masked
    credit_card="4532-1234-5678-9010"  # Will be masked
)
# Output: {..., "password": "***REDACTED***", "credit_card": "****-****-****-****"}
```

---

## Metrics

### What are Metrics?

**Metrics** are numerical measurements of system behavior over time, aggregated into time-series data.

**Key Concept**: Metrics answer "How much?" and "How often?"

### Types of Metrics

#### 1. Counter
A cumulative metric that only increases (or resets to zero).

**Use cases**: Requests count, errors count, jobs completed

```python
from prometheus_client import Counter

http_requests_total = Counter(
    'http_requests_total',
    'Total HTTP requests',
    ['method', 'endpoint', 'status']
)

# Usage
http_requests_total.labels(method='GET', endpoint='/api/users', status='200').inc()
```

#### 2. Gauge
A metric that can go up or down.

**Use cases**: Memory usage, active connections, queue size

```python
from prometheus_client import Gauge

active_users = Gauge('active_users', 'Number of active users')

# Usage
active_users.set(150)  # Set to specific value
active_users.inc()     # Increment
active_users.dec()     # Decrement
```

#### 3. Histogram
Tracks distribution of values and calculates percentiles.

**Use cases**: Request duration, response size

```python
from prometheus_client import Histogram

request_duration = Histogram(
    'http_request_duration_seconds',
    'HTTP request duration',
    ['method', 'endpoint']
)

# Usage
with request_duration.labels(method='GET', endpoint='/api/users').time():
    process_request()  # Automatically times this block
```

**Histogram automatically tracks**:
- Count of observations
- Sum of observed values
- Bucket counts (for percentile calculation)

#### 4. Summary
Similar to histogram but calculates percentiles on the client side.

```python
from prometheus_client import Summary

request_duration = Summary(
    'request_duration_seconds',
    'Request duration in seconds'
)

# Usage
@request_duration.time()
def process_request():
    # Function body
    pass
```

### The Four Golden Signals (Google SRE)

These are the **most important metrics** to monitor for any service:

#### 1. Latency
**Definition**: Time to service a request

**Metrics**:
- Average latency
- P50 (median)
- P95 (95th percentile)
- P99 (99th percentile)
- P99.9

**Why percentiles matter**:
```text
Average latency: 100ms
P50: 80ms   ← 50% of requests faster than this
P95: 200ms  ← 95% of requests faster than this
P99: 500ms  ← 99% of requests faster than this

Problem: Average hides the worst experiences!
```

**Example**:
```python
from prometheus_client import Histogram

request_latency = Histogram(
    'request_duration_seconds',
    'Request latency',
    ['endpoint'],
    buckets=[0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0]  # Define buckets
)

# Usage
import time

start = time.time()
response = handle_request()
duration = time.time() - start

request_latency.labels(endpoint='/api/users').observe(duration)
```

#### 2. Traffic
**Definition**: How much demand is on your system

**Metrics**:
- Requests per second (RPS)
- Messages per second (Kafka)
- Queries per second (QPS)
- Concurrent users

**Example**:
```python
from prometheus_client import Counter

requests_total = Counter(
    'http_requests_total',
    'Total HTTP requests',
    ['method', 'endpoint']
)

def handle_request(method, endpoint):
    requests_total.labels(method=method, endpoint=endpoint).inc()
    # Handle request
```

#### 3. Errors
**Definition**: Rate of failed requests

**Metrics**:
- Error rate (errors/second)
- Error percentage (errors/total requests)
- Error types distribution

**Example**:
```python
from prometheus_client import Counter

errors_total = Counter(
    'http_errors_total',
    'Total HTTP errors',
    ['method', 'endpoint', 'status_code']
)

def handle_request(method, endpoint):
    try:
        response = process_request()
        return response
    except HTTPError as e:
        errors_total.labels(
            method=method,
            endpoint=endpoint,
            status_code=e.status_code
        ).inc()
        raise
```

#### 4. Saturation
**Definition**: How "full" your system is

**Metrics**:
- CPU utilization (%)
- Memory utilization (%)
- Disk I/O (%)
- Network bandwidth (%)
- Database connection pool (used/total)
- Queue depth

**Example**:
```python
from prometheus_client import Gauge
import psutil

cpu_usage = Gauge('cpu_usage_percent', 'CPU usage percentage')
memory_usage = Gauge('memory_usage_percent', 'Memory usage percentage')

def update_system_metrics():
    cpu_usage.set(psutil.cpu_percent())
    memory_usage.set(psutil.virtual_memory().percent)

# Update periodically
import schedule
schedule.every(10).seconds.do(update_system_metrics)
```

### RED Method (Alternative to Golden Signals)

Simpler approach focused on requests:

1. **Rate**: Requests per second
2. **Errors**: Failed requests per second
3. **Duration**: Latency distribution

```python
from prometheus_client import Counter, Histogram

# Rate
request_rate = Counter('requests_total', 'Total requests')

# Errors
error_rate = Counter('errors_total', 'Total errors')

# Duration
request_duration = Histogram('request_duration_seconds', 'Request duration')

def handle_request():
    request_rate.inc()
    
    start = time.time()
    try:
        result = process()
        return result
    except Exception as e:
        error_rate.inc()
        raise
    finally:
        duration = time.time() - start
        request_duration.observe(duration)
```

### USE Method (for Resources)

For infrastructure/resource monitoring:

1. **Utilization**: % time resource is busy
2. **Saturation**: Queue depth, wait time
3. **Errors**: Error count

**Example: Database Connection Pool**
```python
connection_pool_utilization = Gauge(
    'db_pool_utilization',
    'Connection pool utilization'
)

connection_pool_saturation = Gauge(
    'db_pool_queue_length',
    'Connection pool queue length'
)

connection_pool_errors = Counter(
    'db_pool_errors',
    'Connection pool errors'
)

def monitor_connection_pool(pool):
    # Utilization
    used = pool.size() - pool.available()
    utilization = (used / pool.size()) * 100
    connection_pool_utilization.set(utilization)
    
    # Saturation
    connection_pool_saturation.set(pool.queue_length())
    
    # Errors tracked when connection fails
```

### Implementing Metrics with Prometheus

#### Setup in Python (Flask example)

```python
from flask import Flask
from prometheus_client import Counter, Histogram, Gauge, generate_latest
import time

app = Flask(__name__)

# Define metrics
request_count = Counter(
    'http_requests_total',
    'Total HTTP requests',
    ['method', 'endpoint', 'status']
)

request_duration = Histogram(
    'http_request_duration_seconds',
    'HTTP request duration',
    ['method', 'endpoint']
)

active_requests = Gauge(
    'http_requests_active',
    'Active HTTP requests'
)

# Middleware to track metrics
@app.before_request
def before_request():
    request.start_time = time.time()
    active_requests.inc()

@app.after_request
def after_request(response):
    duration = time.time() - request.start_time
    
    request_count.labels(
        method=request.method,
        endpoint=request.path,
        status=response.status_code
    ).inc()
    
    request_duration.labels(
        method=request.method,
        endpoint=request.path
    ).observe(duration)
    
    active_requests.dec()
    
    return response

# Expose metrics endpoint
@app.route('/metrics')
def metrics():
    return generate_latest()

# Your API endpoints
@app.route('/api/users')
def get_users():
    return {"users": []}
```

#### Prometheus Configuration

```yaml
# prometheus.yml
global:
  scrape_interval: 15s  # How often to scrape metrics

scrape_configs:
  - job_name: 'my-app'
    static_configs:
      - targets: ['localhost:5000']  # Your app's /metrics endpoint
```

### Cardinality and Label Best Practices

**Cardinality**: Number of unique time series created by label combinations.

**Problem**: High cardinality = high memory usage

```python
# ❌ BAD: user_id creates millions of time series
request_count = Counter(
    'requests_total',
    'Total requests',
    ['user_id']  # If you have 1M users = 1M time series!
)

# ✅ GOOD: Use aggregatable labels
request_count = Counter(
    'requests_total',
    'Total requests',
    ['method', 'endpoint', 'status']  # Maybe 100 time series total
)
```

**Label Guidelines**:
- ✅ Low cardinality (< 100 values)
- ✅ Fixed set of values
- ✅ Examples: `method`, `status`, `region`, `environment`
- ❌ High cardinality
- ❌ Examples: `user_id`, `request_id`, `session_id`, `email`

### Aggregation and Querying (PromQL)

**PromQL** (Prometheus Query Language) is used to query metrics.

**Common Queries**:

```promql
# Request rate (requests per second)
rate(http_requests_total[5m])

# Error rate
rate(http_errors_total[5m])

# Error percentage
(rate(http_errors_total[5m]) / rate(http_requests_total[5m])) * 100

# P95 latency
histogram_quantile(0.95, rate(http_request_duration_seconds_bucket[5m]))

# CPU usage per service
cpu_usage_percent{service="order-service"}

# Average response time by endpoint
avg(rate(http_request_duration_seconds_sum[5m])) by (endpoint)
```

**Time Ranges**:
- `[5m]` = last 5 minutes
- `[1h]` = last hour
- `[1d]` = last day

### Alerting on Metrics

**Alerting Rules** (alertmanager):

```yaml
groups:
  - name: api_alerts
    rules:
      # High error rate
      - alert: HighErrorRate
        expr: |
          (rate(http_errors_total[5m]) / rate(http_requests_total[5m])) > 0.05
        for: 5m
        labels:
          severity: critical
        annotations:
          summary: "High error rate detected"
          description: "Error rate is {{ $value | humanizePercentage }}"
      
      # High latency
      - alert: HighLatency
        expr: |
          histogram_quantile(0.95, rate(http_request_duration_seconds_bucket[5m])) > 1
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "High P95 latency"
          description: "P95 latency is {{ $value }}s"
      
      # High CPU usage
      - alert: HighCPU
        expr: cpu_usage_percent > 80
        for: 10m
        labels:
          severity: warning
        annotations:
          summary: "High CPU usage"
          description: "CPU usage is {{ $value }}%"
```

---

## Traces

### What is Distributed Tracing?

**Distributed Tracing** tracks a request as it flows through multiple services in a distributed system.

**Key Concepts**:
- **Trace**: The complete journey of a request
- **Span**: A single operation within a trace
- **Context Propagation**: Passing trace information between services

### The Problem Without Tracing

```text
User Request → API Gateway → Auth Service → Order Service → Payment Service → Database
                                                           → Inventory Service

Question: "Why is checkout slow?"

Without tracing:
- Check API Gateway logs
- Check Auth Service logs
- Check Order Service logs
- Check Payment Service logs
- Try to correlate by timestamp (painful!)
- Still unclear which service is slow
```

### With Distributed Tracing

```text
Trace ID: abc-123-def

Span 1: API Gateway      [====] 50ms
  Span 2: Auth Service   [==] 20ms
  Span 3: Order Service  [================] 150ms
    Span 4: Payment      [=] 10ms
    Span 5: Inventory    [============] 120ms  ← Bottleneck found!
      Span 6: Database   [==========] 100ms    ← Root cause!

Total: 200ms
```

**Now you know**: Inventory Service is slow because of slow database queries.

### Trace Structure

```text
Trace (abc-123-def)
│
├─ Span: GET /api/checkout
│   ├─ start_time: 2026-01-22T10:00:00.000Z
│   ├─ end_time: 2026-01-22T10:00:00.200Z
│   ├─ duration: 200ms
│   ├─ service: api-gateway
│   ├─ status: ok
│   │
│   ├─ Child Span: authenticate_user
│   │   ├─ start_time: 2026-01-22T10:00:00.010Z
│   │   ├─ duration: 20ms
│   │   ├─ service: auth-service
│   │   └─ attributes: {user_id: 123}
│   │
│   └─ Child Span: process_order
│       ├─ start_time: 2026-01-22T10:00:00.030Z
│       ├─ duration: 150ms
│       ├─ service: order-service
│       │
│       ├─ Child Span: charge_payment
│       │   ├─ duration: 10ms
│       │   └─ service: payment-service
│       │
│       └─ Child Span: update_inventory
│           ├─ duration: 120ms
│           ├─ service: inventory-service
│           │
│           └─ Child Span: db_query
│               ├─ duration: 100ms
│               └─ attributes: {query: "UPDATE inventory..."}
```

### Implementing Tracing (Manual)

#### Basic Trace Context

```python
import time
import uuid
from contextvars import ContextVar

# Context variable for trace ID (thread-safe)
trace_context = ContextVar('trace_context', default=None)

class Span:
    def __init__(self, name, trace_id=None, parent_span_id=None):
        self.trace_id = trace_id or str(uuid.uuid4())
        self.span_id = str(uuid.uuid4())
        self.parent_span_id = parent_span_id
        self.name = name
        self.start_time = time.time()
        self.end_time = None
        self.attributes = {}
        self.status = "ok"
    
    def set_attribute(self, key, value):
        self.attributes[key] = value
    
    def end(self):
        self.end_time = time.time()
    
    def duration_ms(self):
        if self.end_time:
            return (self.end_time - self.start_time) * 1000
        return None
    
    def __enter__(self):
        trace_context.set(self)
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end()
        if exc_type:
            self.status = "error"
        # Send span to backend (Jaeger, Zipkin, etc.)
        export_span(self)

# Usage
def process_order(order_id):
    with Span("process_order") as span:
        span.set_attribute("order_id", order_id)
        
        # Child spans automatically inherit context
        authenticate_user()
        charge_payment()
        update_inventory()

def authenticate_user():
    parent = trace_context.get()
    with Span("authenticate_user", 
              trace_id=parent.trace_id,
              parent_span_id=parent.span_id):
        # Auth logic
        time.sleep(0.02)  # Simulate work

def charge_payment():
    parent = trace_context.get()
    with Span("charge_payment",
              trace_id=parent.trace_id,
              parent_span_id=parent.span_id):
        # Payment logic
        time.sleep(0.01)
```

### Context Propagation Across Services

**The Challenge**: How does Service B know it's part of Service A's trace?

**Solution**: Propagate trace context in HTTP headers

```python
import requests

def call_downstream_service(trace_id, span_id):
    headers = {
        'X-Trace-ID': trace_id,
        'X-Parent-Span-ID': span_id,
    }
    
    response = requests.get(
        'http://downstream-service/api/endpoint',
        headers=headers
    )
    return response

# Downstream service extracts context
def handle_request(request):
    trace_id = request.headers.get('X-Trace-ID')
    parent_span_id = request.headers.get('X-Parent-Span-ID')
    
    with Span("downstream_operation",
              trace_id=trace_id,
              parent_span_id=parent_span_id):
        # Process request
        pass
```

---

## OpenTelemetry

### What is OpenTelemetry?

**OpenTelemetry (OTel)** is a vendor-neutral standard for observability. It provides:
- Unified APIs for logs, metrics, traces
- Automatic instrumentation
- Context propagation
- Export to any backend (Jaeger, Prometheus, etc.)

**Why OpenTelemetry?**
- ✅ No vendor lock-in
- ✅ Automatic instrumentation for common frameworks
- ✅ Single SDK for all three pillars
- ✅ Industry standard (CNCF project)

### OpenTelemetry Architecture

```text
┌──────────────────────────────────────────┐
│         Your Application                 │
├──────────────────────────────────────────┤
│   OpenTelemetry SDK (Python/Java/Go)     │
│   ├─ Tracing API                         │
│   ├─ Metrics API                         │
│   └─ Logging API (Future)                │
├──────────────────────────────────────────┤
│   OpenTelemetry Collector (Optional)     │
│   ├─ Receive telemetry                   │
│   ├─ Process/Filter/Transform            │
│   └─ Export to backends                  │
└──────────────────────────────────────────┘
            ↓         ↓         ↓
    ┌───────────┬──────────┬──────────┐
    │  Jaeger   │Prometheus│ Backend  │
    │ (Traces)  │(Metrics) │ (Logs)   │
    └───────────┴──────────┴──────────┘
```

### Setting Up OpenTelemetry in Python

#### Installation

```bash
pip install opentelemetry-api
pip install opentelemetry-sdk
pip install opentelemetry-instrumentation-flask  # Auto-instrument Flask
pip install opentelemetry-instrumentation-requests  # Auto-instrument requests
pip install opentelemetry-exporter-jaeger  # Export to Jaeger
```

#### Basic Setup (Manual Instrumentation)

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.jaeger.thrift import JaegerExporter
from opentelemetry.sdk.resources import Resource

# Configure resource (identifies your service)
resource = Resource.create({
    "service.name": "order-service",
    "service.version": "1.0.0",
    "deployment.environment": "production"
})

# Set up tracer provider
trace.set_tracer_provider(TracerProvider(resource=resource))

# Configure exporter (send traces to Jaeger)
jaeger_exporter = JaegerExporter(
    agent_host_name="localhost",
    agent_port=6831,
)

# Add span processor
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(jaeger_exporter)
)

# Get tracer
tracer = trace.get_tracer(__name__)

# Usage
def process_order(order_id):
    with tracer.start_as_current_span("process_order") as span:
        span.set_attribute("order.id", order_id)
        span.set_attribute("user.id", get_user_id())
        
        # Child spans
        authenticate_user()
        charge_payment(order_id)
        
        span.set_status(trace.Status(trace.StatusCode.OK))
        return {"status": "success"}

def charge_payment(order_id):
    with tracer.start_as_current_span("charge_payment") as span:
        span.set_attribute("payment.amount", 99.99)
        try:
            # Payment logic
            result = payment_api.charge()
            return result
        except Exception as e:
            span.set_status(trace.Status(
                trace.StatusCode.ERROR,
                str(e)
            ))
            span.record_exception(e)
            raise
```

#### Automatic Instrumentation (Easier!)

```python
from flask import Flask
from opentelemetry.instrumentation.flask import FlaskInstrumentor
from opentelemetry.instrumentation.requests import RequestsInstrumentor

app = Flask(__name__)

# Automatically instrument Flask
FlaskInstrumentor().instrument_app(app)

# Automatically instrument requests library
RequestsInstrumentor().instrument()

# That's it! All Flask routes and requests calls are now traced

@app.route('/api/users')
def get_users():
    # This is automatically traced
    response = requests.get('http://user-service/users')
    return response.json()
```

### Advanced OpenTelemetry Patterns

#### Custom Span Attributes

```python
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

def process_checkout(cart):
    with tracer.start_as_current_span("process_checkout") as span:
        # Add custom attributes
        span.set_attribute("cart.item_count", len(cart.items))
        span.set_attribute("cart.total_amount", cart.total)
        span.set_attribute("cart.currency", "USD")
        span.set_attribute("user.id", cart.user_id)
        span.set_attribute("user.tier", get_user_tier(cart.user_id))
        
        # Semantic conventions (standard attribute names)
        span.set_attribute("http.method", "POST")
        span.set_attribute("http.route", "/api/checkout")
        
        # Process checkout
        result = charge_customer(cart)
        
        return result
```

#### Span Events

```python
def transfer_money(from_account, to_account, amount):
    with tracer.start_as_current_span("transfer_money") as span:
        span.set_attribute("transfer.amount", amount)
        
        # Record events within the span
        span.add_event("Validating accounts")
        validate_accounts(from_account, to_account)
        
        span.add_event("Checking balance")
        check_balance(from_account, amount)
        
        span.add_event("Executing transfer")
        execute_transfer(from_account, to_account, amount)
        
        span.add_event("Transfer completed", {
            "transaction_id": "txn_123",
            "timestamp": time.time()
        })
```

#### Span Links (Connecting Related Traces)

```python
# Batch processing scenario: one parent job spawns many child jobs

def process_batch(order_ids):
    with tracer.start_as_current_span("process_batch") as batch_span:
        batch_context = trace.get_current_span().get_span_context()
        
        for order_id in order_ids:
            # Each order gets its own trace
            # But linked to the batch trace
            process_single_order(order_id, batch_context)

def process_single_order(order_id, batch_context):
    # Create new trace with link to batch
    with tracer.start_as_current_span(
        "process_order",
        links=[trace.Link(batch_context)]
    ) as span:
        span.set_attribute("order.id", order_id)
        # Process order
```

### OpenTelemetry Collector

The **Collector** is an optional component that receives, processes, and exports telemetry.

**Benefits**:
- Centralized configuration
- Batching and buffering
- Filtering and sampling
- Multiple export destinations
- Offloads work from application

**Collector Configuration** (collector-config.yaml):

```yaml
receivers:
  otlp:
    protocols:
      grpc:
        endpoint: 0.0.0.0:4317
      http:
        endpoint: 0.0.0.0:4318

processors:
  batch:
    timeout: 10s
    send_batch_size: 1024
  
  # Sample traces (keep 10%)
  probabilistic_sampler:
    sampling_percentage: 10

exporters:
  jaeger:
    endpoint: jaeger:14250
    tls:
      insecure: true
  
  prometheus:
    endpoint: 0.0.0.0:8889
  
  logging:
    loglevel: debug

service:
  pipelines:
    traces:
      receivers: [otlp]
      processors: [batch, probabilistic_sampler]
      exporters: [jaeger, logging]
    
    metrics:
      receivers: [otlp]
      processors: [batch]
      exporters: [prometheus, logging]
```

**Send telemetry to collector**:

```python
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

# Instead of Jaeger exporter, use OTLP
otlp_exporter = OTLPSpanExporter(
    endpoint="localhost:4317",
    insecure=True
)

trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(otlp_exporter)
)
```

### Sampling Strategies

**Problem**: High-traffic systems generate too many traces (expensive to store).

**Solutions**:

#### 1. Head Sampling (at trace start)
```python
from opentelemetry.sdk.trace.sampling import TraceIdRatioBased

# Sample 10% of traces
sampler = TraceIdRatioBased(0.1)

trace.set_tracer_provider(TracerProvider(sampler=sampler))
```

#### 2. Tail Sampling (after trace completes)
Collector configuration:
```yaml
processors:
  tail_sampling:
    policies:
      # Keep all error traces
      - name: error-policy
        type: status_code
        status_code:
          status_codes: [ERROR]
      
      # Keep slow traces (> 1s)
      - name: slow-traces
        type: latency
        latency:
          threshold_ms: 1000
      
      # Sample 1% of successful fast traces
      - name: probabilistic-policy
        type: probabilistic
        probabilistic:
          sampling_percentage: 1
```

**Benefits**:
- Keep all errors
- Keep all slow traces
- Sample normal traffic

---

## Debugging Production Failures

### The Production Debugging Challenge

**Key Constraints**:
- ❌ Can't use debugger
- ❌ Can't easily reproduce
- ❌ Limited time (users impacted)
- ❌ Pressure to fix quickly

**Solution**: Systematic approach with observability data

### Step-by-Step Debugging Workflow

#### Step 1: Identify the Problem

**Questions to answer**:
- What is failing? (specific error/symptom)
- When did it start? (timestamp)
- Who is affected? (all users? specific segment?)
- How severe? (% of requests, impact)

**Tools**:
- Monitoring dashboards
- Alerting system
- User reports

**Example**:
```text
Alert: High error rate on /api/checkout
- Started: 2026-01-22 14:30 UTC
- Error rate: 15% (normal: < 1%)
- Affected users: All users
- Severity: Critical
```

#### Step 2: Gather Context with Observability Data

**Metrics** (What's happening):
```promql
# Error rate spike
rate(http_errors_total{endpoint="/api/checkout"}[5m])

# Latency change
histogram_quantile(0.95, rate(http_request_duration_seconds_bucket{endpoint="/api/checkout"}[5m]))

# Traffic pattern
rate(http_requests_total{endpoint="/api/checkout"}[5m])
```

**Logs** (What errors):
```sql
SELECT * FROM logs
WHERE level = 'ERROR'
  AND endpoint = '/api/checkout'
  AND timestamp > '2026-01-22 14:30:00'
LIMIT 100
```

**Traces** (Where it's failing):
- Sample failing requests
- Identify slowest spans
- Find common patterns

#### Step 3: Form Hypothesis

**Based on the data**, create hypotheses:

```text
Observation: P95 latency jumped from 100ms to 5000ms at 14:30

Hypothesis 1: Database connection pool exhausted
- Check: Connection pool metrics
- Expected: Pool utilization = 100%

Hypothesis 2: Downstream service timeout
- Check: External API call duration in traces
- Expected: Timeout errors in logs

Hypothesis 3: Cache invalidation caused database overload
- Check: Cache hit rate metrics
- Expected: Cache miss rate spiked
```

#### Step 4: Validate Hypothesis

**Test each hypothesis**:

```python
# Example: Check connection pool
# Query metrics:
db_pool_utilization{service="order-service"}
# Result: 100% since 14:30 ✓ Confirms hypothesis 1

# Check logs:
# Filter: "connection pool" AND timestamp > 14:30
# Result: "HikariPool-1 - Connection is not available, request timed out after 30000ms"
# ✓ Confirms hypothesis 1
```

#### Step 5: Find Root Cause

**Dig deeper**:

```text
Question: Why is connection pool exhausted?
├─ Check: Connection leak?
│  └─ Trace analysis: All connections properly closed ✗
│
├─ Check: Traffic spike?
│  └─ Metrics: Traffic unchanged ✗
│
├─ Check: Slow queries?
│  └─ Traces show: One query taking 4500ms ✓
│
└─ Root Cause: Slow database query

Deep dive into query:
├─ Query: SELECT * FROM orders WHERE user_id = ? AND created_at > ?
├─ Issue: Missing index on (user_id, created_at)
└─ Fix: Add index
```

#### Step 6: Implement Fix

**Immediate mitigation**:
```sql
-- Add missing index
CREATE INDEX idx_orders_user_created ON orders(user_id, created_at);
```

**Verify fix**:
- Monitor error rate (should decrease)
- Check latency (should return to normal)
- Verify connection pool (should stabilize)

#### Step 7: Post-Mortem and Prevention

**Document**:
1. What happened
2. Impact
3. Root cause
4. Fix applied
5. Prevention measures

**Prevention**:
- Add alerting on connection pool saturation
- Query performance testing in CI
- Database index recommendations in code review

### Real-World Example: Debugging Slow API

**Scenario**: `/api/checkout` endpoint suddenly slow

**Step 1: Metrics**
```promql
# P95 latency
histogram_quantile(0.95, rate(http_request_duration_seconds_bucket{endpoint="/api/checkout"}[5m]))
# Result: 3.2 seconds (was 0.1 seconds)
```

**Step 2: Traces**
Sample trace breakdown:
```text
Trace: checkout request (3.2s total)
├─ authenticate_user: 50ms
├─ validate_cart: 30ms
├─ calculate_taxes: 2.8s ← Bottleneck!
│  └─ external_api_call: 2.7s ← Root cause!
└─ charge_payment: 300ms
```

**Step 3: Logs**
```json
{
  "level": "ERROR",
  "message": "Tax API timeout",
  "service": "checkout-service",
  "duration_ms": 2700,
  "error": "ReadTimeout: Request to tax-api timed out after 3000ms"
}
```

**Step 4: Root Cause**
- Tax calculation service is slow/down
- No timeout configured (defaults to 3s)
- No fallback mechanism

**Step 5: Immediate Fix**
```python
# Add timeout and fallback
try:
    tax = calculate_taxes(cart, timeout=1.0)
except TimeoutError:
    logger.warning("Tax API timeout, using estimated tax")
    tax = estimate_tax(cart)  # Fallback
```

**Step 6: Long-term Fix**
- Implement circuit breaker
- Add caching for tax calculations
- Monitor tax service health

### Common Production Issues and How to Debug

#### Issue 1: Intermittent Timeouts

**Symptoms**:
- Random request timeouts
- No consistent pattern
- Affects < 5% of requests

**Debugging**:
```python
# Add detailed timeout logging
import structlog

logger = structlog.get_logger()

def call_external_api():
    start = time.time()
    try:
        response = requests.get(url, timeout=5.0)
        elapsed = time.time() - start
        logger.info("api_call_success", duration_ms=elapsed*1000)
        return response
    except requests.Timeout:
        elapsed = time.time() - start
        logger.error("api_call_timeout",
                    duration_ms=elapsed*1000,
                    url=url,
                    timeout_configured=5.0)
        raise
```

**Analysis**:
- Look for patterns in timeout logs
- Check network latency metrics
- Examine external service metrics
- Review connection pool settings

#### Issue 2: Memory Leak

**Symptoms**:
- Memory usage grows over time
- Application eventually OOMs
- Performance degrades gradually

**Debugging**:
```python
# Monitor memory over time
from prometheus_client import Gauge
import psutil

memory_usage = Gauge('process_memory_bytes', 'Memory usage')

def update_memory_metrics():
    process = psutil.Process()
    memory_usage.set(process.memory_info().rss)

# Call periodically
```

**Analysis**:
```bash
# Generate memory dump
import objgraph
objgraph.show_most_common_types(limit=20)

# Track growth
objgraph.show_growth()
```

#### Issue 3: Cascading Failures

**Symptoms**:
- One service failure causes others to fail
- Rapid spread of failures
- System-wide outage

**Prevention & Debugging**:

```python
# Implement circuit breaker
from pybreaker import CircuitBreaker

payment_breaker = CircuitBreaker(
    fail_max=5,  # Open after 5 failures
    reset_timeout=60  # Try again after 60s
)

@payment_breaker
def call_payment_service():
    return payment_api.charge()

try:
    result = call_payment_service()
except CircuitBreakerError:
    logger.warning("Payment service circuit open, using fallback")
    result = queue_payment_for_retry()
```

**Circuit Breaker States**:
```text
CLOSED (normal) → failures → OPEN (blocking calls) → timeout → HALF_OPEN (testing) → success → CLOSED
```

---

## SLOs, SLIs, and SLAs

### Understanding the Hierarchy

```text
SLA (Service Level Agreement)
  ↓ (drives)
SLO (Service Level Objective)
  ↓ (measured by)
SLI (Service Level Indicator)
```

### SLI (Service Level Indicator)

**Definition**: A quantitative measure of service level.

**SLIs are metrics** that represent user experience.

**Common SLIs**:

| Category | SLI | Measurement |
|----------|-----|-------------|
| **Availability** | Request success rate | `(successful_requests / total_requests) * 100` |
| **Latency** | Request duration | P50, P95, P99 latency |
| **Quality** | Error rate | `(errors / total_requests) * 100` |
| **Durability** | Data retention | `(successful_backups / total_backups) * 100` |

**Example SLIs for an API**:
```python
# SLI 1: Availability
# Measurement: % of successful HTTP requests (status 200-299)
availability_sli = (successful_requests / total_requests) * 100

# SLI 2: Latency
# Measurement: P95 latency < 200ms
latency_sli = p95_latency_ms

# SLI 3: Error Rate
# Measurement: % of requests returning 5xx errors
error_rate_sli = (server_errors / total_requests) * 100
```

### SLO (Service Level Objective)

**Definition**: Target value or range for an SLI over a time period.

**Format**: "SLI should be [target] over [time window]"

**Example SLOs**:
```text
SLO 1: Availability
- 99.9% of requests should succeed over a 30-day window

SLO 2: Latency
- 95% of requests should complete in < 200ms over a 7-day window

SLO 3: Error Rate
- < 0.1% of requests should return 5xx errors over a 30-day window
```

### Error Budget

**Definition**: Amount of time a service can be unavailable before violating SLO.

**Calculation**:
```text
Error Budget = (1 - SLO) × Time Window

Example:
- SLO: 99.9% availability
- Time window: 30 days
- Error Budget = (1 - 0.999) × 30 days = 0.001 × 30 days
- Error Budget = 43.2 minutes per month
```

**Error Budget in Practice**:
```python
from prometheus_client import Gauge

# Track error budget consumption
error_budget_remaining = Gauge(
    'error_budget_remaining_seconds',
    'Remaining error budget in seconds'
)

def calculate_error_budget():
    # 30-day window
    window_seconds = 30 * 24 * 60 * 60
    
    # 99.9% SLO → 0.1% error budget
    error_budget_seconds = window_seconds * 0.001
    
    # Calculate downtime in last 30 days
    downtime = get_downtime_last_30_days()
    
    # Remaining budget
    remaining = error_budget_seconds - downtime
    error_budget_remaining.set(remaining)
    
    return remaining

# Alert when error budget low
def check_error_budget_alert():
    remaining = calculate_error_budget()
    if remaining < (43.2 * 60 * 0.1):  # < 10% remaining
        send_alert("Error budget critically low!")
```

**Using Error Budget for Decisions**:
```text
Error Budget > 50%:
- ✅ Deploy new features
- ✅ Perform experiments
- ✅ Refactor code

Error Budget < 50%:
- ⚠️ Focus on reliability
- ⚠️ Reduce deployment frequency
- ⚠️ Fix bugs

Error Budget < 10%:
- 🔴 Feature freeze
- 🔴 Emergency reliability work only
- 🔴 All hands on stability
```

### SLA (Service Level Agreement)

**Definition**: Contract with users/customers about expected service level, with consequences if not met.

**SLA includes**:
- SLO targets
- Measurement method
- Exclusions (maintenance windows)
- Consequences (refunds, credits)

**Example SLA**:
```text
API Service SLA

Service Level Objectives:
- Availability: 99.95% uptime per month
- Latency: P95 < 300ms
- Error Rate: < 0.1% server errors

Measurement:
- Calculated from Prometheus metrics
- Excludes scheduled maintenance
- Measured monthly

Consequences:
- 99.95% - 99.9%:   10% service credit
- 99.9% - 99.0%:    25% service credit
- < 99.0%:          50% service credit

Exclusions:
- Scheduled maintenance (< 4 hours/month)
- Customer infrastructure issues
- Force majeure events
```

### SLI/SLO Implementation Example

```python
from prometheus_client import Counter, Histogram, Gauge
import time

# Define SLI metrics
requests_total = Counter(
    'http_requests_total',
    'Total HTTP requests',
    ['status']
)

request_duration = Histogram(
    'http_request_duration_seconds',
    'HTTP request duration',
    buckets=[0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
)

# SLO tracking
slo_availability = Gauge('slo_availability_percent', 'Current availability SLO %')
slo_latency_p95 = Gauge('slo_latency_p95_ms', 'Current P95 latency SLO')

# Middleware to track SLIs
def track_request(endpoint, status_code, duration):
    # Track for availability SLI
    if 200 <= status_code < 300:
        requests_total.labels(status='success').inc()
    elif status_code >= 500:
        requests_total.labels(status='error').inc()
    
    # Track for latency SLI
    request_duration.observe(duration)

# Calculate and publish SLO compliance
def calculate_slo_compliance():
    # Query Prometheus for last 7 days
    # (In practice, use Prometheus API)
    
    # Availability SLO: 99.9%
    total = get_total_requests_7d()
    successful = get_successful_requests_7d()
    availability = (successful / total) * 100
    slo_availability.set(availability)
    
    # Latency SLO: P95 < 200ms
    p95_latency = get_p95_latency_7d()
    slo_latency_p95.set(p95_latency)
    
    return {
        'availability': {
            'target': 99.9,
            'actual': availability,
            'met': availability >= 99.9
        },
        'latency': {
            'target': 200,
            'actual': p95_latency,
            'met': p95_latency <= 200
        }
    }

# Alert on SLO violations
def check_slo_alerts():
    slo_status = calculate_slo_compliance()
    
    if not slo_status['availability']['met']:
        send_alert(
            "Availability SLO violated",
            f"Current: {slo_status['availability']['actual']:.2f}%",
            f"Target: {slo_status['availability']['target']}%"
        )
    
    if not slo_status['latency']['met']:
        send_alert(
            "Latency SLO violated",
            f"Current P95: {slo_status['latency']['actual']:.0f}ms",
            f"Target P95: {slo_status['latency']['target']}ms"
        )
```

### Alerting Strategy Based on SLOs

**Burn Rate**: How fast you're consuming error budget.

```text
Example:
- Monthly error budget: 43.2 minutes
- Current downtime rate: 0.5% (instead of 0.1% target)
- Burn rate: 5× (consuming budget 5× faster than acceptable)
- Projected: Error budget exhausted in 6 days instead of 30
```

**Multi-Window Alerting**:

```yaml
# Alert on fast burn rate (immediate issue)
- alert: SLOFastBurn
  expr: |
    (
      sum(rate(http_errors_total[5m])) / sum(rate(http_requests_total[5m]))
    ) > 0.05  # 5% error rate = 50× burn rate for 99.9% SLO
  for: 5m
  labels:
    severity: critical
  annotations:
    summary: "Fast error budget burn (critical)"

# Alert on slow burn rate (trending issue)
- alert: SLOSlowBurn
  expr: |
    (
      sum(rate(http_errors_total[1h])) / sum(rate(http_requests_total[1h]))
    ) > 0.002  # 0.2% error rate = 2× burn rate
  for: 1h
  labels:
    severity: warning
  annotations:
    summary: "Slow error budget burn (warning)"
```

**Why multi-window**:
- Fast burn (5m): Catches severe outages
- Slow burn (1h): Catches gradual degradation

---

## Observability Best Practices

### 1. Start with SLOs

**Process**:
1. Define what matters to users
2. Choose SLIs that represent user experience
3. Set realistic SLO targets
4. Instrument systems to measure SLIs
5. Build dashboards and alerts around SLOs

**Example User Journey**:
```text
User wants to: Check out and purchase items

Critical SLIs:
1. Can user add items to cart? (Availability)
2. Can user view cart? (Availability)
3. Can user complete checkout? (Availability + Latency)
4. Does payment succeed? (Availability + Error Rate)

SLOs:
- Checkout availability: 99.9%
- Checkout latency P95: < 2 seconds
- Payment error rate: < 0.5%
```

### 2. Use Structured Logging

**Benefits**:
- Easy to query
- Machine-readable
- Context preserved
- Aggregatable

**Implementation**:
```python
import structlog

logger = structlog.get_logger()

# ✅ Good: Structured with context
logger.info(
    "payment_processed",
    order_id=12345,
    user_id=67890,
    amount=99.99,
    payment_method="credit_card",
    transaction_id="txn_abc123"
)

# ❌ Bad: Unstructured string
logger.info("Payment of $99.99 processed for order 12345")
```

### 3. Implement Distributed Tracing Early

**Why**:
- Microservices are complex without tracing
- Hard to add later
- Critical for debugging

**Minimal Implementation**:
```python
# Just add OpenTelemetry auto-instrumentation
from opentelemetry.instrumentation.flask import FlaskInstrumentor
from opentelemetry.instrumentation.requests import RequestsInstrumentor

FlaskInstrumentor().instrument_app(app)
RequestsInstrumentor().instrument()
```

### 4. Monitor the Four Golden Signals

**Always track**:
1. Latency
2. Traffic
3. Errors
4. Saturation

**Dashboard Example**:
```text
Service: order-service

┌─────────────────────────────────────┐
│ Latency                             │
│ P50: 45ms   P95: 120ms   P99: 250ms│
├─────────────────────────────────────┤
│ Traffic                             │
│ 1,234 req/s  (↑ 15% vs 1h ago)     │
├─────────────────────────────────────┤
│ Errors                              │
│ 0.2% error rate  (12 errors/min)   │
├─────────────────────────────────────┤
│ Saturation                          │
│ CPU: 45%  Memory: 60%  Pool: 70%   │
└─────────────────────────────────────┘
```

### 5. Correlation Across Pillars

**Connect logs, metrics, and traces**:

```python
# When logging, include trace_id
from opentelemetry import trace

def handle_request():
    span = trace.get_current_span()
    trace_id = span.get_span_context().trace_id
    
    logger.info(
        "request_processed",
        trace_id=format(trace_id, '032x'),  # Include trace ID in logs
        order_id=12345
    )
```

**Query workflow**:
1. Metrics: "Error rate spike at 14:30"
2. Logs: "Find error logs at 14:30" → Get trace_id
3. Traces: "View trace abc-123" → See full request flow

### 6. Alert on Symptoms, Not Causes

**❌ Bad: Alert on causes**
```yaml
- alert: HighCPU
  expr: cpu_usage > 80%
```
**Problem**: High CPU might not impact users.

**✅ Good: Alert on symptoms**
```yaml
- alert: HighLatency
  expr: p95_latency > 500ms
```
**Benefit**: This definitely impacts users.

**Then investigate**: CPU, memory, database, etc.

### 7. Reduce Alert Fatigue

**Problems**:
- Too many alerts
- Noisy alerts
- False positives

**Solutions**:

#### Group Related Alerts
```yaml
# Don't alert on every pod restart
# Alert when multiple pods restart
- alert: HighPodRestartRate
  expr: rate(pod_restarts_total[5m]) > 0.1
```

#### Use Alert Severity
```yaml
alerts:
  critical:  # Page on-call
    - Service completely down
    - Data loss